In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

rain = pd.read_csv("/content/rainfall.csv")
temp = pd.read_csv("/content/temp.csv")
yield_data = pd.read_csv("/content/yield.csv")

rain.columns = rain.columns.str.strip()
temp.columns = temp.columns.str.strip()
yield_data.columns = yield_data.columns.str.strip()

rain.rename(columns={"average_rain_fall_mm_per_year": "Rainfall"}, inplace=True)
rain["Rainfall"] = pd.to_numeric(rain["Rainfall"], errors="coerce")

temp.rename(columns={"year": "Year", "country": "Area", "avg_temp": "Temp"}, inplace=True)
temp["Temp"] = pd.to_numeric(temp["Temp"], errors="coerce")

yield_data.rename(columns={"Value": "Yield"}, inplace=True)
yield_data["Yield"] = pd.to_numeric(yield_data["Yield"], errors="coerce")

if "Item" in yield_data.columns:
    yield_data.rename(columns={"Item": "Crop"}, inplace=True)

yield_data["Crop"] = yield_data["Crop"].str.strip().str.lower()
yield_data["Area"] = yield_data["Area"].str.strip()
rain["Area"] = rain["Area"].str.strip()
temp["Area"] = temp["Area"].str.strip()

data = yield_data.copy()
data = data.merge(rain[["Area", "Year", "Rainfall"]], on=["Area", "Year"], how="left")
data = data.merge(temp[["Area", "Year", "Temp"]], on=["Area", "Year"], how="left")

data = data.dropna(subset=["Yield", "Rainfall", "Temp", "Crop", "Area"])

le_area = LabelEncoder()
le_crop = LabelEncoder()
data["Area"] = le_area.fit_transform(data["Area"])
data["Crop"] = le_crop.fit_transform(data["Crop"])

X = data[["Area", "Crop", "Rainfall", "Temp"]]
y = data["Yield"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R² Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

print(pd.DataFrame({"Actual": y_test[:5].values, "Predicted": y_pred[:5]}))


R² Score: 0.9260922772111401
MAE: 10311.688898254592
   Actual      Predicted
0   90385   94678.823333
1  227606  175549.483333
2  196487  290838.900000
3   54035   61214.944000
4    2793    3958.740000
